# Clase 8 — QAOA y optimización combinatoria

## Objetivos de aprendizaje

- Traducir un problema de optimización (Max-Cut) a un Hamiltoniano de coste
- Construir y entrenar un circuito QAOA
- Muestrear el circuito entrenado para obtener soluciones concretas

Muchos problemas del mundo real son problemas de optimización combinatoria: repartir rutas de reparto, agrupar clientes, diseñar redes de telecomunicaciones. En esta clase trabajas con uno de los problemas de referencia en este campo, Max-Cut, y aplicas el Algoritmo Cuántico Aproximado de Optimización (QAOA) para resolverlo con un circuito entrenable.

In [ ]:
!pip install -q cirq tensorflow tensorflow-quantum networkx

## El problema Max-Cut

Max-Cut parte de un grafo: un conjunto de nodos conectados por aristas. Tu tarea consiste en dividir los nodos en dos grupos de forma que el número de aristas "cortadas" (con un nodo en cada grupo) sea el máximo posible. Es un problema combinatorio clásico y, con QAOA, lo abordas con un ordenador cuántico NISQ (con ruido y sin corrección de errores, como viste en el capítulo dos).

## Traducir el problema a un Hamiltoniano de coste

La idea central de QAOA consiste en traducir tu problema de optimización a un Hamiltoniano de coste: un operador cuántico cuyo valor esperado, al medirlo sobre un estado concreto, coincide con cuántas aristas corta esa división de nodos.

Para Max-Cut, construyes ese Hamiltoniano sumando un término `Z_i · Z_j` por cada arista entre los nodos i y j: ese término vale -1 si los dos nodos están en el mismo grupo, y +1 si están en grupos distintos. Cuanto mayor sea el valor esperado del Hamiltoniano, mejor es el corte que representa.

## El circuito QAOA: alternar coste y mezcla

El circuito arranca poniendo todos los qubits en superposición con puertas Hadamard, representando así todas las divisiones posibles a la vez. Después alterna dos bloques, cada uno con su propio parámetro entrenable:

- El Hamiltoniano de coste, parametrizado por un ángulo que llamas γ (gamma): empuja el estado hacia las divisiones que cortan más aristas
- El Hamiltoniano de mezcla, parametrizado por un ángulo que llamas β (beta): mezcla el estado para que el circuito explore otras divisiones además de la que ya parece buena, evitando quedarse atascado en una solución mediocre

Cuantas más veces repites ese ciclo coste-mezcla (parámetro `p` en la literatura de QAOA), más cerca puede llegar el circuito de la solución óptima, a costa de un circuito más profundo. Recuerda el capítulo dos: más profundidad significa más exposición a la decoherencia.

## De la teoría a Cirq: `tfq.util.exponential`

Cirq y TFQ no te obligan a descomponer a mano cada Hamiltoniano en puertas cuánticas elementales. La función `tfq.util.exponential(operadores, coeficientes)` construye directamente el circuito que aplica `exp(-i · coeficiente · operador)`, justo la operación que necesitas para aplicar un Hamiltoniano durante un tiempo controlado por el parámetro entrenable.

## Leer la respuesta: muestrear el circuito entrenado

Tras entrenar los parámetros γ y β, mides el circuito muchas veces con `tfq.layers.Sample`. Cada medida te da una posible división de los nodos en dos grupos; la división que más se repite (o la de mayor corte entre las muestras) es tu solución aproximada al problema Max-Cut.

A continuación tienes el ejemplo completo comentado, con un grafo pequeño (un ciclo de seis nodos) para que el entrenamiento sea rápido en una sesión de clase.

In [ ]:
import tensorflow as tf
import tensorflow_quantum as tfq
import cirq
import sympy
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx

# Grafo ciclo de seis nodos: 0-1-2-3-4-5-0
grafo = nx.cycle_graph(6)
nx.draw_networkx(grafo)
plt.title("Grafo de partida: ciclo de seis nodos")
plt.show()

qubits = [cirq.GridQubit(0, i) for i in range(grafo.number_of_nodes())]

# Estado inicial: todos los qubits en superposición (todas las divisiones a la vez)
estado_inicial = cirq.Circuit()
for qubit in qubits:
    estado_inicial.append(cirq.H(qubit))

In [ ]:
# Hamiltoniano de mezcla: suma de X sobre cada qubit
hamiltoniano_mezcla = sum(cirq.PauliString(cirq.X(qubits[i])) for i in range(len(qubits)))

# Hamiltoniano de coste: un término Z_i . Z_j por cada arista del grafo
hamiltoniano_coste = grafo.number_of_edges() / 2
for nodo_a, nodo_b in grafo.edges():
    hamiltoniano_coste += cirq.PauliString(0.5 * cirq.Z(qubits[nodo_a]) * cirq.Z(qubits[nodo_b]))

In [ ]:
p = 1  # número de repeticiones del ciclo coste-mezcla
gamma, beta = sympy.symbols("gamma beta")

circuito_qaoa = cirq.Circuit()
circuito_qaoa += tfq.util.exponential(
    operators=[hamiltoniano_coste, hamiltoniano_mezcla],
    coefficients=[gamma, beta],
)

entradas = tfq.convert_to_tensor([estado_inicial])
capa_entrada = tf.keras.layers.Input(shape=(), dtype=tf.dtypes.string)
salida = tfq.layers.PQC(circuito_qaoa, hamiltoniano_coste)(capa_entrada)

modelo_qaoa = tf.keras.models.Model(inputs=capa_entrada, outputs=salida)
modelo_qaoa.compile(loss=tf.keras.losses.MAE, optimizer=tf.keras.optimizers.Adam())

In [ ]:
# Entrenas para que el valor esperado del corte se acerque al máximo teórico (etiqueta ficticia 0)
historial = modelo_qaoa.fit(entradas, np.array([0]), epochs=200)

plt.plot(historial.history["loss"])
plt.title("QAOA — pérdida durante el entrenamiento")
plt.xlabel("Iteración")
plt.ylabel("Pérdida")
plt.show()

Con la pérdida cerca de cero, los parámetros γ y β ya están entrenados. Toca muestrear el circuito para leer divisiones concretas y compararlas con el corte máximo real, que en un grafo tan pequeño puedes calcular por fuerza bruta.

In [ ]:
parametros_entrenados = modelo_qaoa.trainable_variables
circuito_con_datos = tfq.layers.AddCircuit()(entradas, append=circuito_qaoa)
muestras = tfq.layers.Sample()(
    circuito_con_datos,
    symbol_names=[gamma, beta],
    symbol_values=parametros_entrenados,
    repetitions=1000,
)

cortes_obtenidos = []
for bits in muestras.values:
    grupo_uno = [i for i, bit in enumerate(bits) if bit == 1]
    cortes_obtenidos.append(nx.algorithms.cuts.cut_size(grafo, grupo_uno))

corte_maximo = max(
    nx.algorithms.cuts.cut_size(grafo, list(combo))
    for r in range(grafo.number_of_nodes() + 1)
    for combo in __import__("itertools").combinations(grafo.nodes(), r)
)

print("Corte medio obtenido:", np.mean(cortes_obtenidos))
print("Mejor corte posible (fuerza bruta):", corte_maximo)

Con un grafo tan pequeño puedes calcular el corte máximo real por fuerza bruta y comparar: verás que el corte medio que da QAOA se acerca bastante a ese máximo, aunque casi nunca lo alcanza de forma perfecta con solo una repetición (`p = 1`).

## Resumen de puntos clave

- QAOA traduce un problema de optimización combinatoria (Max-Cut) a un Hamiltoniano de coste, cuyo valor esperado mide la calidad de una solución
- El circuito alterna un bloque de coste (parametrizado por γ) y un bloque de mezcla (parametrizado por β), repetidos `p` veces
- `tfq.util.exponential` construye directamente el circuito que aplica un Hamiltoniano durante un tiempo dado por un parámetro entrenable
- Tras entrenar, muestreas el circuito muchas veces para obtener soluciones concretas al problema
- Aumentar `p` (más repeticiones del ciclo coste-mezcla) suele mejorar la solución, a costa de un circuito más profundo